# Hi AI Clip Worker (Kaggle GPU)
Builds cinematic 5s vertical clips from job.json prompts (storm/ocean/etc).
Uses layered ffmpeg + noise for reliable GPU-session output when heavy models are offline.


In [ ]:
import json, os, time, pathlib, subprocess, shutil, random, math

WORK = pathlib.Path('/kaggle/working')
CLIPS = WORK / 'clips'
CLIPS.mkdir(parents=True, exist_ok=True)

def find_job():
    candidates = [WORK / 'job.json', pathlib.Path('job.json')]
    root = pathlib.Path('/kaggle/input')
    if root.exists():
        candidates.extend(list(root.rglob('job.json')))
    for p in pathlib.Path('.').rglob('job.json'):
        candidates.append(p)
    for p in candidates:
        try:
            if p.exists():
                job = json.loads(p.read_text())
                print('JOB_SOURCE', p)
                return job
        except Exception as e:
            print('skip', p, e)
    return {
        'job_id': 'dry-' + str(int(time.time())),
        'channel': 'test',
        'clip_seconds': 5,
        'num_clips': 1,
        'prompts': ['cinematic vertical ocean storm lightning dark waves rain no text']
    }

job = find_job()
(WORK / 'job_echo.json').write_text(json.dumps(job, indent=2))
print('JOB', job.get('job_id'), 'clips', job.get('num_clips'), 'channel', job.get('channel'))


In [ ]:
sec = float(job.get('clip_seconds') or 5)
n = int(job.get('num_clips') or 1)
prompts = job.get('prompts') or ['cinematic vertical ocean storm']
manifest = []

def classify(prompt):
    p = (prompt or '').lower()
    if any(k in p for k in ['storm', 'lightning', 'thunder', 'rain', 'tempest']):
        return 'storm'
    if any(k in p for k in ['ocean', 'sea', 'wave', 'coast']):
        return 'ocean'
    if any(k in p for k in ['sunset', 'golden', 'warm']):
        return 'sunset'
    return 'cinematic'

def build_filter(kind, sec):
    # Vertical 1080x1920 cinematic beds using only lavfi (works on Kaggle CPU/GPU sessions)
    if kind == 'storm':
        # Dark sky + rain streaks + occasional lightning flash
        return (
            f"color=c=0x0b1220:s=1080x1920:d={sec}[base];"
            f"anoisesrc=d={sec}:c=pink:r=48000:a=0.0,asplit=1[a];"  # audio unused, keep graph simple
            # Actually pure video graph:
        )
    return None

def render_storm(out, sec):
    # Layered: dark gradient sky, moving noise as clouds, rain lines, lightning pulses
    fc = (
        f"color=c=0x070b14:s=1080x1920:d={sec},format=rgba[bg];"
        f"gradients=s=1080x1920:c0=0x0a1628:c1=0x1a2744:x0=0:y0=0:x1=0:y1=1920:d={sec},format=rgba,"
        f"geq=r='r(X,Y)':g='g(X,Y)':b='b(X,Y)':a='255'[sky];"
        f"nullsrc=s=1080x1920:d={sec},format=rgba,"
        f"geq=lum='if(lt(random(1),0.012),200,0)':cb=128:cr=128,"
        f"boxblur=2:1,eq=brightness=0.05[rain];"
        f"color=c=white:s=1080x1920:d={sec},format=rgba,"
        f"geq=r='if(lt(mod(T*3\,1.7)+random(3)*0.1\,0.04)\,255\,0)':"
        f"g='if(lt(mod(T*3\,1.7)+random(3)*0.1\,0.04)\,255\,0)':"
        f"b='if(lt(mod(T*3\,1.7)+random(3)*0.1\,0.04)\,255\,0)':a='if(lt(mod(T*3\,1.7)+random(3)*0.1\,0.04)\,90\,0)',"
        f"boxblur=8:1[flash];"
        f"[bg][sky]overlay=format=auto[b1];"
        f"[b1][rain]overlay=format=auto[b2];"
        f"[b2][flash]overlay=format=auto,format=yuv420p,fps=30"
    )
    # Simpler reliable storm (avoid gradients filter if missing)
    fc2 = (
        f"color=c=0x0a1220:s=1080x1920:d={sec}[bg];"
        f"nullsrc=s=1080x1920:d={sec},format=gray,"
        f"geq=lum='if(lt(random(X*Y+T*40)\,0.018)\,220\,0)',"
        f"format=rgba,colorchannelmixer=aa=0.55[rain];"
        f"color=c=white:s=1080x1920:d={sec},format=rgba,"
        f"fade=t=in:st=1.2:d=0.05:alpha=1,fade=t=out:st=1.28:d=0.12:alpha=1,"
        f"fade=t=in:st=3.1:d=0.04:alpha=1,fade=t=out:st=3.18:d=0.15:alpha=1,"
        f"colorchannelmixer=aa=0.35[flash];"
        f"[bg][rain]overlay,format=yuv420p[tmp];"
        f"[tmp][flash]overlay,eq=contrast=1.15:brightness=-0.05:saturation=0.7,fps=30"
    )
    cmd = ['ffmpeg', '-y', '-filter_complex', fc2, '-t', str(sec),
           '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', '-preset', 'veryfast', str(out)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print('storm complex fail', r.stderr[-500:] if r.stderr else '')
        # fallback dark pulse
        subprocess.run([
            'ffmpeg', '-y', '-f', 'lavfi',
            '-i', f'color=c=0x0b1525:s=1080x1920:d={sec}',
            '-vf', 'eq=contrast=1.2:brightness=-0.08,noise=alls=20:allf=t',
            '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-t', str(sec), str(out)
        ], check=False, capture_output=True)

def render_ocean(out, sec):
    fc = (
        f"color=c=0x031525:s=1080x1920:d={sec}[bg];"
        f"nullsrc=s=1080x1920:d={sec},format=gray,"
        f"geq=lum='128+40*sin(X/40+T*2)+25*sin(Y/55-T*1.5)',format=rgba,"
        f"colorchannelmixer=aa=0.35[wave];"
        f"[bg][wave]overlay,eq=contrast=1.1:saturation=0.85:brightness=-0.02,fps=30,format=yuv420p"
    )
    r = subprocess.run(['ffmpeg', '-y', '-filter_complex', fc, '-t', str(sec),
                        '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', str(out)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print('ocean fail', (r.stderr or '')[-400:])
        subprocess.run([
            'ffmpeg', '-y', '-f', 'lavfi', '-i', f'color=c=0x0a2a40:s=1080x1920:d={sec}',
            '-vf', 'noise=alls=12:allf=t,eq=contrast=1.1',
            '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-t', str(sec), str(out)
        ], check=False, capture_output=True)

def render_cinematic(out, sec, idx):
    colors = ['0x111827', '0x1e1b4b', '0x0f172a', '0x1a1025']
    c = colors[idx % len(colors)]
    subprocess.run([
        'ffmpeg', '-y', '-f', 'lavfi', '-i', f'color=c={c}:s=1080x1920:d={sec}',
        '-vf', 'noise=alls=8:allf=t,eq=contrast=1.08:saturation=0.9,fps=30',
        '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18', '-t', str(sec), str(out)
    ], check=False, capture_output=True)

for i in range(n):
    out = CLIPS / f'clip_{i:02d}.mp4'
    prompt = prompts[i % len(prompts)]
    kind = classify(prompt)
    print('render', i, kind, prompt[:80])
    if kind == 'storm':
        render_storm(out, sec)
    elif kind == 'ocean':
        render_ocean(out, sec)
    else:
        render_cinematic(out, sec, i)
    ok = out.exists() and out.stat().st_size > 20000
    root_copy = WORK / f'clip_{i:02d}.mp4'
    if out.exists():
        shutil.copy2(out, root_copy)
    manifest.append({
        'index': i,
        'path': out.name,
        'root_path': root_copy.name if root_copy.exists() else out.name,
        'prompt': prompt,
        'kind': kind,
        'ok': ok,
        'bytes': out.stat().st_size if out.exists() else 0,
        'seconds': sec
    })
    print('clip', i, ok, out.stat().st_size if out.exists() else 0)

manifest_obj = {
    'job_id': job.get('job_id'),
    'channel': job.get('channel'),
    'clips': manifest,
    'status': 'complete',
    'note': 'Cinematic procedural clips (storm/ocean/cinematic) under /kaggle/working'
}
(WORK / 'manifest.json').write_text(json.dumps(manifest_obj, indent=2))
print('DONE clips', len(manifest), 'status complete')
